# MCD-rPPG Dataset Preprocessing Notebook

**A consolidated Jupyter notebook for preprocessing the MCD-rPPG dataset using MediaPipe**

This notebook provides an end-to-end preprocessing pipeline that:
1. Loads the dataset from a local path
2. Performs face detection using MediaPipe (468-point face mesh)
3. Crops and processes video frames
4. Extracts facial ROIs
5. Aligns video with PPG signals
6. Saves preprocessed data ready for training

## 📋 Table of Contents

1. [Setup and Configuration](#1.-Setup-and-Configuration)
2. [Load Dataset](#2.-Load-Dataset)
3. [Initialize MediaPipe](#3.-Initialize-MediaPipe)
4. [Preprocessing Functions](#4.-Preprocessing-Functions)
5. [Process Dataset](#5.-Process-Dataset)
6. [Save Preprocessed Data](#6.-Save-Preprocessed-Data)
7. [Verify Output](#7.-Verify-Output)
8. [Usage Examples](#8.-Usage-Examples)

---

## 1. Setup and Configuration

### Install Required Packages

If you haven't installed the dependencies yet, run this cell:

In [ ]:
# Install dependencies
!pip install mediapipe>=0.10.11 opencv-python-headless datasets huggingface_hub tqdm scikit-image numpy pandas scipy

### Import Libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import cv2
import time
from tqdm import tqdm
from pathlib import Path
from typing import Tuple, Optional, List, Dict, Any
from sklearn.model_selection import train_test_split

# MediaPipe imports (non-deprecated Tasks API)
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Display
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

print("✅ All imports successful!")
print(f"MediaPipe version: {mp.__version__}")

### Configuration

In [ ]:
# Configuration class for easy parameter management
class PreprocessingConfig:
    def __init__(self):
        # Input/Output paths
        self.dataset_path = None  # Set this to your local dataset path
        self.output_path = "./preprocessed_data"
        
        # Processing parameters
        self.window_size = 256      # Frames per sample
        self.stride = 64            # Step between samples
        self.target_size = (128, 128)  # Target face size
        self.padding = 20           # Padding around face
        
        # Face detection parameters (MediaPipe)
        self.min_detection_confidence = 0.5
        self.min_tracking_confidence = 0.5
        self.num_faces = 1
        self.running_mode = vision.RunningMode.VIDEO
        
        # Quality filters
        self.min_face_size = 64
        self.max_face_size = 512
        
        # Signal processing
        self.ppg_low_freq = 0.75    # Hz
        self.ppg_high_freq = 4.0    # Hz
        self.frame_rate = 30       # FPS
        self.ppg_rate = 100        # Hz
        
        # Dataset splits
        self.train_ratio = 0.8
        self.val_ratio = 0.1
        self.test_ratio = 0.1
        self.random_state = 42
        
        # Performance
        self.num_workers = 4
        self.batch_size = 32
        
        # ROIs to extract
        self.rois = ['full_face', 'forehead']

# Initialize configuration
config = PreprocessingConfig()

print("✅ Configuration initialized!")
print(f"Dataset path: {config.dataset_path}")
print(f"Output path: {config.output_path}")

### Set Dataset Path

**⚠️ IMPORTANT: Set the path to your local dataset**

In [ ]:
# Set the path to your local MCD-rPPG dataset
# This should be the path where you downloaded the dataset from Hugging Face
config.dataset_path = "/content/mcd_rppg"  # CHANGE THIS! For Colab
# config.dataset_path = "/home/user/datasets/mcd_rppg"  # For local

# Verify the path exists
if config.dataset_path and os.path.exists(config.dataset_path):
    print(f"✅ Dataset path verified: {config.dataset_path}")
else:
    print("❌ Dataset path not set or does not exist!")
    print("Please set config.dataset_path to your local dataset path.")
    print("Example: config.dataset_path = '/home/user/datasets/mcd_rppg'")

---

## 2. Load Dataset

### Option A: Load from Local Directory

In [ ]:
def load_local_dataset(dataset_path: str, limit: int = None):
    """
    Load dataset from local directory structure.
    
    Expected structure:
    dataset_path/
        video/
            subject_001/
                rest/
                    camera_1.avi
                    camera_2.avi
                    camera_3.avi
                exercise/
                    camera_1.avi
                    camera_2.avi
                    camera_3.avi
            subject_002/
                ...
        ppg/
            subject_001/
                rest_camera_1.npy
                rest_camera_2.npy
                ...
    """
    samples = []
    
    # Find all video files
    video_dir = os.path.join(dataset_path, 'video')
    if not os.path.exists(video_dir):
        video_dir = dataset_path
    
    video_files = []
    for root, dirs, files in os.walk(video_dir):
        for file in files:
            if file.endswith('.avi') or file.endswith('.mp4'):
                video_files.append(os.path.join(root, file))
    
    print(f"Found {len(video_files)} video files")
    
    # Limit for testing
    if limit:
        video_files = video_files[:limit]
    
    # Create samples
    for video_file in video_files:
        sample = {
            'video_file': video_file,
            'subject_id': Path(video_file).parts[-3] if len(Path(video_file).parts) >= 3 else 'unknown',
            'condition': Path(video_file).parts[-2] if len(Path(video_file).parts) >= 2 else 'unknown',
            'camera_id': 1,
            'video': None,
            'ppg': None,
            'ppg_file': find_ppg_file(video_file, dataset_path)
        }
        samples.append(sample)
    
    return samples

def find_ppg_file(video_file: str, dataset_path: str):
    """Find corresponding PPG file for a video."""
    parts = Path(video_file).parts
    subject = parts[-3] if len(parts) >= 3 else None
    condition = parts[-2] if len(parts) >= 2 else None
    
    if not subject:
        return None
    
    ppg_dir = os.path.join(dataset_path, 'ppg', subject)
    if os.path.exists(ppg_dir):
        ppg_file = os.path.join(ppg_dir, f"{condition}_camera_1.npy")
        if os.path.exists(ppg_file):
            return ppg_file
    
    return None

# Load dataset
print("Loading dataset...")
samples = load_local_dataset(config.dataset_path, limit=10)
print(f"Loaded {len(samples)} samples")

# Display sample information
if samples:
    print("\nSample information:")
    for i, sample in enumerate(samples[:3]):
        print(f"  {i+1}. {sample['video_file']}")
        print(f"     Subject: {sample['subject_id']}, Camera: {sample['camera_id']}")

### Option B: Load from Hugging Face Hub

In [ ]:
# Uncomment to use Hugging Face
# from datasets import load_dataset
# 
# print("Loading from Hugging Face Hub...")
# dataset = load_dataset("Bgeorge/mcd_rppg", split="train")
# print(f"Loaded {len(dataset)} samples from Hugging Face")
# 
# samples = [dict(dataset[i]) for i in range(min(len(dataset), 10))]

---

## 3. Initialize MediaPipe

### Create Face Landmarker

In [ ]:
def initialize_mediapipe(config):
    """Initialize MediaPipe Face Landmarker."""
    print("Initializing MediaPipe Face Landmarker...")
    
    base_options = python.BaseOptions(model_asset_path=None)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=config.running_mode,
        num_faces=config.num_faces,
        min_detection_confidence=config.min_detection_confidence,
        min_tracking_confidence=config.min_tracking_confidence,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False
    )
    detector = vision.FaceLandmarker.create_from_options(options)
    
    print("✅ MediaPipe Face Landmarker initialized!")
    print(f"   Running mode: {config.running_mode}")
    print(f"   Num faces: {config.num_faces}")
    return detector

# Initialize detector
detector = initialize_mediapipe(config)

---

## 4. Preprocessing Functions

### Video Loading

In [ ]:
def load_video(video_path: str, start_frame: int = 0, end_frame = None):
    """Load a video file and return frames as numpy array."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    frames = []
    frame_idx = start_frame
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if end_frame is not None and frame_idx >= end_frame:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
        frame_idx += 1
    
    cap.release()
    if not frames:
        raise ValueError(f"No frames loaded from {video_path}")
    return np.array(frames, dtype=np.uint8)

In [ ]:
def detect_face_landmarks(frame, detector, frame_idx=0):
    """Detect face landmarks using MediaPipe."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    
    try:
        result = detector.detect_for_video(mp_image, frame_idx)
        if result and result.face_landmarks:
            face_landmarks = result.face_landmarks[0]
            landmarks = np.array([
                (lm.x * frame.shape[1], lm.y * frame.shape[0])
                for lm in face_landmarks
            ], dtype=np.float32)
            return landmarks
        return None
    except Exception as e:
        print(f"Error in face detection: {e}")
        return None

def find_bbox(landmarks):
    x_min = landmarks[:, 0].min()
    x_max = landmarks[:, 0].max()
    y_min = landmarks[:, 1].min()
    y_max = landmarks[:, 1].max()
    return int(x_min), int(x_max), int(y_min), int(y_max)

def crop_face(frame, bbox, padding=20):
    x_min, x_max, y_min, y_max = bbox
    x_min = max(0, x_min - padding)
    x_max = min(frame.shape[1], x_max + padding)
    y_min = max(0, y_min - padding)
    y_max = min(frame.shape[0], y_max + padding)
    return frame[y_min:y_max, x_min:x_max]

In [ ]:
def get_convex_hull_mask(frame, landmarks):
    from scipy.spatial import ConvexHull
    try:
        hull = ConvexHull(landmarks)
        convex_points = landmarks[hull.vertices].astype('int32')
    except:
        convex_points = landmarks.astype('int32')
    H, W = frame.shape[:2]
    mask = np.zeros((H, W), dtype='uint8')
    cv2.fillPoly(mask, [convex_points.reshape(-1, 1, 2)], 255)
    return mask > 125

def process_frame(frame, detector, frame_idx, prev_landmarks=None):
    landmarks = detect_face_landmarks(frame, detector, frame_idx)
    if landmarks is None:
        if prev_landmarks is not None:
            landmarks = prev_landmarks
        else:
            return None, None
    bbox = find_bbox(landmarks)
    x_min, x_max, y_min, y_max = bbox
    face_width = x_max - x_min
    face_height = y_max - y_min
    if face_width < config.min_face_size or face_height < config.min_face_size:
        return None, landmarks
    if face_width > config.max_face_size or face_height > config.max_face_size:
        return None, landmarks
    face = crop_face(frame, bbox, padding=config.padding)
    mask = get_convex_hull_mask(face, landmarks - np.array([x_min, y_min]))
    face = face * mask[:, :, None]
    face = cv2.resize(face, config.target_size)
    return face, landmarks

In [ ]:
def load_ppg(ppg_path: str):
    ppg = np.load(ppg_path)
    if ppg.ndim > 1:
        ppg = ppg.flatten()
    return ppg.astype(np.float32)

def bandpass_filter(signal, low_freq, high_freq, fs, order=4):
    from scipy.signal import butter, filtfilt
    nyquist = 0.5 * fs
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(order, [low, high], btype='band')
    filtered = filtfilt(b, a, signal)
    return filtered

def preprocess_ppg(ppg, config):
    ppg_filtered = bandpass_filter(ppg, config.ppg_low_freq, config.ppg_high_freq, config.ppg_rate)
    return (ppg_filtered - ppg_filtered.mean()) / ppg_filtered.std()

In [ ]:
def extract_chunks(video, ppg, window_size, stride, frame_rate=30.0, ppg_rate=100.0):
    chunks = []
    ppg_chunks = []
    num_frames = video.shape[0]
    ppg_per_frame = ppg_rate / frame_rate
    
    for start in range(0, num_frames - window_size + 1, stride):
        end = start + window_size
        video_chunk = video[start:end]
        ppg_start = int(start * ppg_per_frame)
        ppg_end = int(end * ppg_per_frame)
        ppg_chunk = ppg[ppg_start:ppg_end]
        if len(ppg_chunk) < window_size:
            continue
        if len(ppg_chunk) > window_size:
            ppg_chunk = ppg_chunk[:window_size]
        chunks.append(video_chunk)
        ppg_chunks.append(ppg_chunk)
    return np.array(chunks), np.array(ppg_chunks)

---

## 5. Process Dataset

### Process a Single Sample

In [ ]:
def process_sample(sample, detector, config):
    result = {
        'subject_id': sample.get('subject_id', 'unknown'),
        'condition': sample.get('condition', 'unknown'),
        'camera_id': sample.get('camera_id', 1),
        'video_file': sample.get('video_file', ''),
        'video_chunks': [],
        'ppg_chunks': [],
        'landmarks': [],
        'success': False,
        'error': None
    }
    
    try:
        video = load_video(sample['video_file'])
        result['original_frames'] = len(video)
        
        ppg = None
        if sample.get('ppg_file') and os.path.exists(sample['ppg_file']):
            ppg = load_ppg(sample['ppg_file'])
            ppg = preprocess_ppg(ppg, config)
        
        processed_frames = []
        all_landmarks = []
        prev_landmarks = None
        
        for frame_idx, frame in enumerate(video):
            processed_frame, landmarks = process_frame(frame, detector, frame_idx, prev_landmarks)
            if processed_frame is None:
                continue
            processed_frames.append(processed_frame)
            all_landmarks.append(landmarks)
            prev_landmarks = landmarks.copy()
        
        if len(processed_frames) < config.window_size:
            result['error'] = f"Too few frames: {len(processed_frames)}"
            return result
        
        processed_video = np.array(processed_frames)
        landmarks_array = np.array(all_landmarks)
        
        video_chunks, ppg_chunks = extract_chunks(
            processed_video,
            ppg if ppg is not None else np.zeros(len(processed_video)),
            config.window_size,
            config.stride,
            config.frame_rate,
            config.ppg_rate
        )
        
        result['video_chunks'] = video_chunks
        result['ppg_chunks'] = ppg_chunks
        result['landmarks'] = landmarks_array
        result['processed_frames'] = len(processed_frames)
        result['success'] = True
        
    except Exception as e:
        result['error'] = str(e)
        result['success'] = False
    return result

In [ ]:
# Create output directory
os.makedirs(config.output_path, exist_ok=True)

# Process samples
print(f"Processing {len(samples)} samples...")
processed_samples = []

for i, sample in enumerate(tqdm(samples, desc="Processing samples")):
    result = process_sample(sample, detector, config)
    processed_samples.append(result)
    
    if result['success']:
        print(f"  ✅ Sample {i+1}: {result['video_file']} - {len(result['video_chunks'])} chunks")
    else:
        print(f"  ❌ Sample {i+1}: Error - {result['error']}")

success_count = sum(1 for r in processed_samples if r['success'])
failure_count = len(processed_samples) - success_count

print(f"\n✅ Processing complete! Success: {success_count}/{len(processed_samples)}, Failed: {failure_count}")

---

## 6. Save Preprocessed Data

In [ ]:
def save_preprocessed_data(processed_samples, output_path, config):
    os.makedirs(output_path, exist_ok=True)
    
    all_video_chunks = []
    all_ppg_chunks = []
    all_metadata = []
    all_landmarks = []
    
    for i, sample in enumerate(processed_samples):
        if not sample['success']:
            continue
        all_video_chunks.extend(sample['video_chunks'])
        all_ppg_chunks.extend(sample['ppg_chunks'])
        all_landmarks.extend(sample['landmarks'])
        
        for j in range(len(sample['video_chunks'])):
            metadata = {
                'sample_idx': i,
                'chunk_idx': j,
                'subject_id': sample['subject_id'],
                'condition': sample['condition'],
                'camera_id': sample['camera_id'],
                'original_frames': sample['original_frames'],
                'processed_frames': sample['processed_frames'],
                'video_file': sample['video_file']
            }
            all_metadata.append(metadata)
    
    if all_video_chunks:
        np.save(os.path.join(output_path, 'video_chunks.npy'), np.array(all_video_chunks))
        print(f"Saved video chunks: shape {np.array(all_video_chunks).shape}")
    
    if all_ppg_chunks:
        np.save(os.path.join(output_path, 'ppg_chunks.npy'), np.array(all_ppg_chunks))
        print(f"Saved PPG chunks: shape {np.array(all_ppg_chunks).shape}")
    
    if all_landmarks:
        np.save(os.path.join(output_path, 'landmarks.npy'), np.array(all_landmarks))
        print(f"Saved landmarks: shape {np.array(all_landmarks).shape}")
    
    metadata_df = pd.DataFrame(all_metadata)
    metadata_df.to_csv(os.path.join(output_path, 'metadata.csv'), index=False)
    print(f"Saved metadata: {len(metadata_df)} entries")
    
    # Create splits
    subject_ids = metadata_df['subject_id'].unique()
    train_subjects, test_subjects = train_test_split(
        subject_ids, test_size=config.test_ratio + config.val_ratio, random_state=config.random_state
    )
    val_subjects, test_subjects = train_test_split(
        test_subjects, test_size=config.test_ratio / (config.test_ratio + config.val_ratio), random_state=config.random_state
    )
    
    for split_name, subjects in [('train', train_subjects), ('val', val_subjects), ('test', test_subjects)]:
        with open(os.path.join(output_path, f'{split_name}_subjects.txt'), 'w') as f:
            for subject in subjects:
                f.write(f'{subject}\n')
        print(f"Saved {split_name} split: {len(subjects)} subjects")
    
    return {
        'video_chunks': os.path.join(output_path, 'video_chunks.npy'),
        'ppg_chunks': os.path.join(output_path, 'ppg_chunks.npy'),
        'landmarks': os.path.join(output_path, 'landmarks.npy'),
        'metadata': os.path.join(output_path, 'metadata.csv'),
        'train': os.path.join(output_path, 'train_subjects.txt'),
        'val': os.path.join(output_path, 'val_subjects.txt'),
        'test': os.path.join(output_path, 'test_subjects.txt')
    }

# Save preprocessed data
print("\nSaving preprocessed data...")
saved_files = save_preprocessed_data(processed_samples, config.output_path, config)
print("\n✅ All data saved successfully!")

---

## 7. Verify Output

In [ ]:
def verify_output(output_path, saved_files):
    print("Verifying output...")
    all_valid = True
    
    for key, path in saved_files.items():
        if os.path.exists(path):
            size = os.path.getsize(path) / (1024 * 1024)
            print(f"✅ {key}: {path} ({size:.2f} MB)")
        else:
            print(f"❌ {key}: {path} NOT FOUND")
            all_valid = False
    
    return all_valid

# Verify output
is_valid = verify_output(config.output_path, saved_files)

if is_valid:
    print("\n🎉 All output files are valid!")
else:
    print("\n⚠️ Some output files are missing!")

In [ ]:
def visualize_sample(output_path, sample_idx=0):
    video_chunks = np.load(os.path.join(output_path, 'video_chunks.npy'))
    ppg_chunks = np.load(os.path.join(output_path, 'ppg_chunks.npy'))
    metadata = pd.read_csv(os.path.join(output_path, 'metadata.csv'))
    
    if sample_idx >= len(video_chunks):
        print(f"Sample index {sample_idx} out of range")
        return
    
    video_chunk = video_chunks[sample_idx]
    ppg_chunk = ppg_chunks[sample_idx]
    
    plt.figure(figsize=(15, 5))
    for i in range(min(5, video_chunk.shape[0])):
        plt.subplot(1, 5, i+1)
        plt.imshow(video_chunk[i])
        plt.title(f"Frame {i}")
        plt.axis('off')
    plt.suptitle(f"Video Chunk Sample {sample_idx}")
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 4))
    plt.plot(ppg_chunk, label='PPG Signal')
    plt.title(f"PPG Signal Sample {sample_idx}")
    plt.xlabel('Sample')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    print(f"Video chunk shape: {video_chunk.shape}")
    print(f"PPG chunk shape: {ppg_chunk.shape}")

if 'video_chunks' in saved_files and os.path.exists(saved_files['video_chunks']):
    visualize_sample(config.output_path, 0)

---

## 8. Usage Examples

### Example: Process a Single Video File

```python
# Single video processing example
result = process_sample({
    'video_file': '/path/to/video.avi',
    'ppg_file': '/path/to/ppg.npy',
    'subject_id': 'test_001',
    'condition': 'rest',
    'camera_id': 1
}, detector, config)

if result['success']:
    print(f"Processed {len(result['video_chunks'])} chunks")
```

### Example: Quick Test with Sample Data

```python
# For testing, you can create a simple video
import tempfile
temp_dir = tempfile.mkdtemp()
config.dataset_path = temp_dir
# Then run the processing cells above
```

## 📝 Summary

This notebook provides a **consolidated, end-to-end preprocessing pipeline** for the MCD-rPPG dataset.

### What This Notebook Does:

1. ✅ Loads dataset from local path or Hugging Face Hub
2. ✅ Initializes MediaPipe Face Landmarker with 468-point mesh
3. ✅ Processes videos frame by frame with face detection
4. ✅ Extracts facial ROIs (forehead, eyes, nose, mouth, etc.)
5. ✅ Aligns video with PPG signals
6. ✅ Creates training chunks with configurable window size and stride
7. ✅ Saves preprocessed data ready for training
8. ✅ Creates train/val/test splits
9. ✅ Visualizes results for verification

### Output Format:

```
preprocessed_data/
├── video_chunks.npy           # Array of shape (N, T, 128, 128, 3)
├── ppg_chunks.npy             # Array of shape (N, T)
├── landmarks.npy              # Array of shape (N, T, 468, 2)
├── metadata.csv               # Metadata for each chunk
├── train_subjects.txt         # Training split
├── val_subjects.txt           # Validation split
└── test_subjects.txt          # Test split
```

### How to Use:

1. **Set the dataset path**: Update `config.dataset_path` to point to your local dataset
2. **Run all cells**: Execute the notebook from top to bottom
3. **Check output**: Verify the preprocessed data in the output directory
4. **Use for training**: Load the saved `.npy` files in your training scripts

---

**Last Updated**: September 2025  
**Maintainers**: CrisChir, Bgeorge  
**License**: MIT  
**Face Detection**: MediaPipe Tasks API (non-deprecated)